In [2]:
import torch
import torch.nn as nn
import numpy as np

class HighFrequencyGatedSkip(nn.Module):
    """
    Applies a 2D FFT spectral high-pass mask to Encoder features before Skip Concatenation.
    Decouples structural boundary signals from low-frequency semantic noise.
    """
    def __init__(self, channels: int, cutoff_ratio: float = 0.25):
        super().__init__()
        self.cutoff_ratio = cutoff_ratio
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, encoder_feat: torch.Tensor, decoder_feat: torch.Tensor) -> torch.Tensor:
        # 1. Convert Spatial Encoder Features to Spectral Domain via 2D RFFT
        fft_feat = torch.fft.rfft2(encoder_feat, norm="backward")
        h, w = fft_feat.shape[-2:]

        # 2. Construct High-Pass Spectral Mask (Zero-out DC & Low Frequencies)
        mask = torch.ones((h, w), device=encoder_feat.device)
        cutoff_h, cutoff_w = int(h * self.cutoff_ratio), int(w * self.cutoff_ratio)
        mask[:cutoff_h, :cutoff_w] = 0.0  # Completely filter out low frequency background

        # 3. Inverse FFT to restore High-Frequency Edge Spatial Logits
        high_pass_fft = fft_feat * mask
        high_freq_spatial = torch.fft.irfft2(high_pass_fft, s=encoder_feat.shape[-2:], norm="backward")

        # 4. Context-Aware Gating from Decoder
        gate = self.spatial_gate(decoder_feat)
        filtered_encoder_feat = high_freq_spatial * gate

        # 5. Precise Edge Signal + Decoder Feature Concatenation
        return torch.cat([decoder_feat, filtered_encoder_feat], dim=1)


# =====================================================================
# Hypothesis Proof Metric: Boundary Signal-to-Noise Ratio (SNR) Test
# =====================================================================
B, C, H, W = 2, 16, 64, 64

# 1. 합성 데이터 생성: 매끄러운 저주파 배경(Sin wave) + 선명한 고주파 경계(Vertical Line)
grid_y, grid_x = torch.meshgrid(torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij')
low_freq_bg = torch.sin(2 * np.pi * grid_x) * torch.sin(2 * np.pi * grid_y) # 저주파 노이즈
high_freq_edge = (torch.abs(grid_x - 0.5) < 0.02).float()                    # 고주파 경계 신호

synthetic_enc = (low_freq_bg + high_freq_edge).unsqueeze(0).unsqueeze(0).repeat(B, C, 1, 1)
synthetic_dec = torch.ones_like(synthetic_enc)

# 2. 기존 Skip(단순 Concat) vs HF-Gated Skip 비교 연산
standard_concat_passed = synthetic_enc  # 기존: 저주파/고주파 모두 전달
hf_skip_module = HighFrequencyGatedSkip(channels=C)
hf_skip_passed = hf_skip_module(synthetic_enc, synthetic_dec)[:, C:, :, :] # 제안: 고주파 필터링 후 전달

# 3. 정량적 지표 계산 (Edge 영역 Power / Background 영역 Power = SNR)
edge_mask = (high_freq_edge > 0.5).unsqueeze(0).unsqueeze(0).repeat(B, C, 1, 1)
bg_mask = (high_freq_edge <= 0.5).unsqueeze(0).unsqueeze(0).repeat(B, C, 1, 1)

std_edge_pwr = standard_concat_passed[edge_mask].pow(2).mean().item()
std_bg_pwr = standard_concat_passed[bg_mask].pow(2).mean().item()
std_snr = std_edge_pwr / (std_bg_pwr + 1e-8)

hf_edge_pwr = hf_skip_passed[edge_mask].pow(2).mean().item()
hf_bg_pwr = hf_skip_passed[bg_mask].pow(2).mean().item()
hf_snr = hf_edge_pwr / (hf_bg_pwr + 1e-8)

# 4. 결론 출력
print("=" * 60)
print("[CONCLUSION] High-Frequency Gated Skip Verification Results")
print("=" * 60)
print(f"Standard Concat -> Edge Power: {std_edge_pwr:.4f} | BG Power: {std_bg_pwr:.4f} | SNR: {std_snr:.2f}")
print(f"HF-Gated Skip   -> Edge Power: {hf_edge_pwr:.4f} | BG Power: {hf_bg_pwr:.4f} | SNR: {hf_snr:.2f}")
print("-" * 60)
print(f"-> Boundary Signal-to-Noise Ratio Improved by +{(hf_snr / std_snr - 1) * 100:.1f}%")
print("=" * 60)

[CONCLUSION] High-Frequency Gated Skip Verification Results
Standard Concat -> Edge Power: 1.0012 | BG Power: 0.2500 | SNR: 4.00
HF-Gated Skip   -> Edge Power: 0.1126 | BG Power: 0.0339 | SNR: 3.32
------------------------------------------------------------
-> Boundary Signal-to-Noise Ratio Improved by +-17.1%
